In [13]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))
from paths import DATA, MODELS

OUT = DATA   

In [14]:
df = pd.read_csv(DATA / 'df_model.csv')
df = df.drop(columns=['Unnamed: 0'], errors='ignore')
print(f"Loaded: {df.shape}")

Loaded: (156000, 31)


In [15]:
leakage_cols = ['goals_scored', 'assists', 'bonus', 'clean_sheets',
                'goals_conceded', 'own_goals', 'penalties_missed',
                'penalties_saved', 'red_cards', 'yellow_cards',
                'starts', 'minutes', 'creativity', 'defensive_contribution']
df = df.drop(columns=[c for c in leakage_cols if c in df.columns])

In [16]:
train = df[df['season'] < 2026]
val   = df[df['season'] == 2026]
 
X_train = train.drop(columns=['total_points'])
y_train = train['total_points']
X_val   = val.drop(columns=['total_points'])
y_val   = val['total_points']
 
print(f"Train: {X_train.shape} | Val: {X_val.shape}")
print(f"Features: {X_train.columns.tolist()}")

Train: (133647, 16) | Val: (22353, 16)
Features: ['position', 'value', 'GW', 'season', 'team', 'was_home', 'player_id', 'opponent_team_name', 'avg_minutes_last3', 'avg_starts_last3', 'avg_points_last3', 'avg_points_vs_opponent', 'avg_goals_conceded_opponent', 'avg_clean_sheets_opponent', 'avg_goals_conceded_away', 'avg_goals_conceded_home']


##Linear Regeression

In [17]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_val)
print(f"\nLinear Regression — MAE: {mean_absolute_error(y_val, y_pred_lr):.2f} | RMSE: {root_mean_squared_error(y_val, y_pred_lr):.2f}")


Linear Regression — MAE: 1.22 | RMSE: 2.21


In [18]:
search_rf = RandomizedSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    {'n_estimators':[100,200], 'max_depth':[5,10,20,None],
     'min_samples_leaf':[1,2,4], 'max_features':['sqrt','log2']},
    n_iter=20, cv=3, scoring='neg_mean_absolute_error',
    random_state=42, n_jobs=-1, verbose=1)
search_rf.fit(X_train, y_train)
y_pred_rf = search_rf.predict(X_val)
print(f"Random Forest     — MAE: {mean_absolute_error(y_val, y_pred_rf):.2f} | RMSE: {root_mean_squared_error(y_val, y_pred_rf):.2f}")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Random Forest     — MAE: 1.06 | RMSE: 2.13


In [19]:
search_xgb = RandomizedSearchCV(
    XGBRegressor(random_state=42, n_jobs=-1),
    {'n_estimators':[100,200,300], 'max_depth':[3,5,7],
     'learning_rate':[0.01,0.05,0.1], 'subsample':[0.7,0.8,1.0],
     'colsample_bytree':[0.7,0.8,1.0]},
    n_iter=20, cv=3, scoring='neg_mean_absolute_error',
    random_state=42, n_jobs=-1, verbose=1)
search_xgb.fit(X_train, y_train)
y_pred_xgb = search_xgb.predict(X_val)
print(f"XGBoost           — MAE: {mean_absolute_error(y_val, y_pred_xgb):.2f} | RMSE: {root_mean_squared_error(y_val, y_pred_xgb):.2f}")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
XGBoost           — MAE: 1.09 | RMSE: 2.21


In [20]:
search_lgbm = RandomizedSearchCV(
    LGBMRegressor(random_state=42, n_jobs=-1),
    {'n_estimators':[100,200,300], 'max_depth':[3,5,7],
     'learning_rate':[0.01,0.05,0.1], 'num_leaves':[31,50,70],
     'subsample':[0.7,0.8,1.0]},
    n_iter=20, cv=3, scoring='neg_mean_absolute_error',
    random_state=42, n_jobs=-1, verbose=1)
search_lgbm.fit(X_train, y_train)
y_pred_lgbm = search_lgbm.predict(X_val)
print(f"LightGBM          — MAE: {mean_absolute_error(y_val, y_pred_lgbm):.2f} | RMSE: {root_mean_squared_error(y_val, y_pred_lgbm):.2f}")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000808 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1084
[LightGBM] [Info] Number of data points in the train set: 133647, number of used features: 16
[LightGBM] [Info] Start training from score 1.189761
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightG

In [21]:
search_lasso = RandomizedSearchCV(
    Lasso(), {'alpha':[0.001,0.01,0.1,1.0,10.0]},
    n_iter=5, cv=3, scoring='neg_mean_absolute_error', random_state=42, verbose=1)
search_lasso.fit(X_train, y_train)
y_pred_lasso = search_lasso.predict(X_val)
print(f"Lasso             — MAE: {mean_absolute_error(y_val, y_pred_lasso):.2f} | RMSE: {root_mean_squared_error(y_val, y_pred_lasso):.2f}")

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Lasso             — MAE: 1.21 | RMSE: 2.18


In [22]:
search_ridge = RandomizedSearchCV(
    Ridge(), {'alpha':[0.001,0.01,0.1,1.0,10.0,100.0]},
    n_iter=6, cv=3, scoring='neg_mean_absolute_error', random_state=42, verbose=1)
search_ridge.fit(X_train, y_train)
y_pred_ridge = search_ridge.predict(X_val)
print(f"Ridge             — MAE: {mean_absolute_error(y_val, y_pred_ridge):.2f} | RMSE: {root_mean_squared_error(y_val, y_pred_ridge):.2f}")

Fitting 3 folds for each of 6 candidates, totalling 18 fits
Ridge             — MAE: 1.22 | RMSE: 2.21


In [23]:
ensemble = VotingRegressor([
    ('lr',    lr),
    ('lasso', search_lasso.best_estimator_),
    ('ridge', search_ridge.best_estimator_),
    ('xgb',   search_xgb.best_estimator_),
    ('lgbm',  search_lgbm.best_estimator_)
])
ensemble.fit(X_train, y_train)
y_pred_ens = ensemble.predict(X_val)
print(f"Ensemble          — MAE: {mean_absolute_error(y_val, y_pred_ens):.2f} | RMSE: {root_mean_squared_error(y_val, y_pred_ens):.2f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000876 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1084
[LightGBM] [Info] Number of data points in the train set: 133647, number of used features: 16
[LightGBM] [Info] Start training from score 1.189761
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

In [24]:
MODELS.mkdir(exist_ok=True)
joblib.dump(search_rf, MODELS / 'fpl_model.pkl')
joblib.dump(ensemble, MODELS / 'fpl_ensemble.pkl')
print("\nModels saved!")
print(f"Features used: {X_train.columns.tolist()}")


Models saved!
Features used: ['position', 'value', 'GW', 'season', 'team', 'was_home', 'player_id', 'opponent_team_name', 'avg_minutes_last3', 'avg_starts_last3', 'avg_points_last3', 'avg_points_vs_opponent', 'avg_goals_conceded_opponent', 'avg_clean_sheets_opponent', 'avg_goals_conceded_away', 'avg_goals_conceded_home']


#After trying some models and some ensembles , lasso linear and ridge performed the best , even better then ensemble , i will use linear 